In [ ]:
# 데이터 처리 및 분석
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # maxOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
    
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

merge_df = pd.read_csv("../../Data/merged_df_260325.csv")
trans = pd.read_csv("../../Data/transactions_260325.csv")

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [36]:
df_core = merge_df[merge_df['offer_label'].notna()].copy()

received_cnt = df_core[df_core['event'] == 'received'].groupby('offer_label')['person'].count()
viewed_cnt = df_core[df_core['event'] == 'viewed'].groupby('offer_label')['person'].count()

completed_events = df_core[df_core['event'] == 'completed'][['person', 'offer_label', 'time']].rename(columns={'time': 'completed_time'})
# 같은 사람이 같은 오퍼를 여러번 본 경우 처음 봤을 때만 남김
viewed_events = df_core[df_core['event'] == 'viewed'][['person', 'offer_label', 'time']].rename(columns={'time': 'viewed_time'})
viewed_events = viewed_events.groupby(['person', 'offer_label'])['viewed_time'].min().reset_index()


# 같은 사람과 같은 오퍼를 기준으로 viewed와 completed 연결
true_comp_df = pd.merge(completed_events, viewed_events, on=['person', 'offer_label'], how='left')
true_comp_df = true_comp_df[
    (true_comp_df['viewed_time'].notna()) &
    (true_comp_df['viewed_time'] <= true_comp_df['completed_time'])
]
true_completed_cnt = true_comp_df.groupby('offer_label')['person'].count()
true_completions_for_rev = true_comp_df[['person', 'completed_time', 'offer_label']].rename(columns={'completed_time': 'time'})


# 오퍼 완료한 순간의 거래만 가져옴 (해찬님 코드 참고)
matched_sales = pd.merge(trans, true_completions_for_rev, on=['person', 'time'], how='inner')

# 오퍼별 총 매출 집계
revenue_df = matched_sales.groupby('offer_label').agg(
    total_offer_revenue=('amount', 'sum')
)


final_kpi_df = pd.DataFrame({
    'received_cnt': received_cnt,
    'viewed_cnt': viewed_cnt,
    'true_completed_cnt': true_completed_cnt
}).fillna(0).join(revenue_df).fillna(0)

final_kpi_df['true_completed_rate'] = (final_kpi_df['true_completed_cnt'] / final_kpi_df['received_cnt'] * 100).round(2)
final_kpi_df['revenue_per_send'] = (final_kpi_df['total_offer_revenue'] / final_kpi_df['received_cnt']).round(2)

### 피어슨 상관계수

In [44]:
from scipy.stats import pearsonr

corr, p_value = pearsonr(final_kpi_df['true_completed_rate'],final_kpi_df['revenue_per_send'])

print(f'- 상관관계 값: {corr}')
print(f'- p-value 값: {p_value}')

- 상관관계 값: 0.9712008985507516
- p-value 값: 2.906734201520214e-06


- 상관관계가 0.97로 상당히 강한 양의 상관관계를 나타냄
: 두 변수는 비례 관계
- p-value가 작기 때문에 통계적으로 유의미함

= 두 변수는 서로 밀접하게 연관되어 있음